<a href="https://colab.research.google.com/github/Themix12/Large-Language-Models-and-Generative-AI-lab-sheet-ID-34970---10965220/blob/main/lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

text = """
the cat sat on the mat.
the dog ran in the park.
the cat saw the dog.
the dog sat near the cat.
""" * 200

# Character-level tokenizer
chars = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Dataset tokens:", len(data))


Vocabulary size: 19
Dataset tokens: 19400


In [ ]:
block_size = 32

def get_batch(batch_size=16):
    starts = torch.randint(
        len(data) - block_size - 1, (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size] for i in starts
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1] for i in starts
    ])

    return x, y

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, block_size,
                 embed_dim=32, num_heads=4):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size, embed_dim
        )
        self.position_embedding = nn.Embedding(
            block_size, embed_dim
        )

        self.attention = nn.MultiheadAttention(
            embed_dim, num_heads, batch_first=True
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, embed_dim)
        )

        self.output = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape

        positions = torch.arange(T, device=x.device)
        h = (
            self.token_embedding(x)
            + self.position_embedding(positions)
        )

        # Prevent each position from seeing future tokens
        mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool),
            diagonal=1
        )

        a, _ = self.attention(
            h, h, h, attn_mask=mask,
            need_weights=False
        )

        h = self.norm1(h + a)
        h = self.norm2(h + self.ffn(h))

        return self.output(h)

model = TinyTransformer(vocab_size, block_size)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.001
)

for step in range(1000):
    x, y = get_batch()

    logits = model(x)

    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        y.reshape(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}: loss = {loss.item():.4f}")

@torch.no_grad()
def generate(prompt, max_new_chars=100):
    model.eval()

    ids = torch.tensor(
        [encode(prompt)], dtype=torch.long
    )

    for _ in range(max_new_chars):
        x = ids[:, -block_size:]
        logits = model(x)

        # Probabilities for the next character
        probs = F.softmax(logits[:, -1, :], dim=-1)

        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)

    return decode(ids[0].tolist())

print(generate("the cat"))

print(generate("the dog"))

Step 0: loss = 3.1343
Step 100: loss = 1.0845
Step 200: loss = 0.7635
Step 300: loss = 0.4837
Step 400: loss = 0.2851
Step 500: loss = 0.1584
Step 600: loss = 0.1273
Step 700: loss = 0.1076
Step 800: loss = 0.1026
Step 900: loss = 0.0859
the cat.

the cat sat on the cat sat on the mat.
the dog ran in the park.
the cat saw the dog.
the dog sat 
the dog.
the dog sat nsat the mat.
tsathe done dog ran the dog r rk.
the park.
the dog.
the dog san n the d
